# 01. Разведочный анализ

**Цель блокнота** — познакомиться с данными до любых выводов: сколько их, какого они качества, как устроены.

Здесь мы ничего не доказываем. Наблюдения из этого блокнота — это гипотезы, которые проверяются в `02_analysis.ipynb`.

**Данные:** 344 пингвина трёх видов с архипелага Палмер (Антарктида), Horst, Hill, Gorman (2020). Файл скачивается и очищается конвейером `python -m palmer_penguins.pipeline`.

In [ ]:
# Расширение autoreload перезагружает изменённые модули проекта перед каждой ячейкой.
# Без него правки в src/palmer_penguins/*.py не были бы видны до перезапуска ядра.
%load_ext autoreload
%autoreload 2

import pandas as pd  # таблицы
import seaborn as sns  # статистические графики поверх matplotlib

# Функции и пути из нашего пакета: загрузка «сырых» и очищенных данных,
# корень проекта, от которого считаются все пути.
from palmer_penguins.data import PROJECT_ROOT, load_clean, load_raw

# Папка для картинок. Оператор / у Path склеивает части пути
# с правильным разделителем для любой ОС.
FIG = PROJECT_ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)  # создать папку, если её ещё нет

## 1. Сырые данные: что лежит в файле

Начинаем с **неочищенных** данных: важно увидеть проблемы источника своими глазами, прежде чем их исправлять.

In [ ]:
# Исходный CSV из data/raw — ровно в том виде, в каком он скачан.
raw = load_raw()

# info() показывает для каждого столбца тип и число НЕпустых значений.
# Первая проверка: числа прочитались как числа (float64/int64), а не как текст.
raw.info()

In [ ]:
# isna() даёт таблицу True/False («значение пропущено?»),
# sum() по столбцам считает True как 1 — получаем число пропусков в каждом столбце.
raw.isna().sum()

**Наблюдение.** У 2 птиц нет ни одного измерения (по 2 пропуска в каждом числовом столбце — это одни и те же две строки), а пол неизвестен у 11. Отсюда решения в функции `clean`: строки без измерений удаляются, пропуски пола остаются и отбрасываются только там, где пол нужен.

In [ ]:
# Сколько птиц каждого вида: баланс классов.
# Пригодится в 02 и 03: доля самого частого вида — это точность «глупой» модели,
# а неравные доли — причина делить данные со стратификацией.
raw["species"].value_counts()

## 2. Очищенные данные: как устроены виды

Дальше работаем с очищенной таблицей (342 строки). Её создаёт конвейер, а `load_clean` читает её и восстанавливает категориальные типы столбцов.

In [ ]:
df = load_clean()

# Средние размеры по видам.
# observed=True — показывать только встречающиеся категории
# (значение по умолчанию менялось между версиями pandas, поэтому указываем явно).
# round(1) — округление до одного знака, чтобы таблица читалась.
df.groupby("species", observed=True)[
    ["bill_length_mm", "bill_depth_mm", "body_mass_g"]
].mean().round(1)

**Наблюдение.** Виды отличаются по-разному:
- у **Gentoo** клюв длинный (≈47.5 мм) и неглубокий (≈15.0 мм), и сами они самые тяжёлые (≈5.1 кг);
- у **Adelie** клюв короткий (≈38.8 мм) и глубокий (≈18.3 мм);
- **Chinstrap** — длинный (≈48.8 мм) **и** глубокий (≈18.4 мм) клюв.

Похоже, что по двум размерам клюва виды хорошо различимы — это вопрос 3 в `02_analysis`.

In [ ]:
# Таблица сопряжённости: сколько птиц каждого вида на каждом острове.
# Строки — виды, столбцы — острова, в клетках — количество.
pd.crosstab(df["species"], df["island"])

**Важное ограничение.** Gentoo живут только на острове Biscoe, Chinstrap — только на Dream. Вид и остров **перепутаны**: по этим данным эффект острова нельзя отделить от эффекта вида. Утверждение «пингвины с Biscoe тяжелее» на самом деле говорило бы о Gentoo.

## 3. Все признаки на одной картинке

`pairplot` рисует диаграмму рассеяния для каждой пары числовых признаков, а на диагонали — оценку плотности распределения каждого признака отдельно для каждого вида.

In [ ]:
# hue="species" — раскрасить точки по виду.
# vars — какие столбцы включить (иначе попал бы и год наблюдения).
g = sns.pairplot(
    df,
    hue="species",
    vars=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"],
)

# Сохраняем картинку в reports/figures, чтобы вставить в README.
g.savefig(FIG / "pairplot.png", dpi=100)

## Итоги разведки

1. Данные почти чистые: 2 строки без измерений и 11 пропусков пола.
2. Классы не сбалансированы: Adelie 152, Gentoo 124, Chinstrap 68.
3. Вид и остров перепутаны — выводы об островах по этим данным делать нельзя.
4. На диаграмме «длина × глубина клюва» три вида образуют три почти раздельных облака — гипотеза для вопроса 3.
5. Внутри каждого облака длина и глубина клюва, кажется, растут вместе, а по всем птицам сразу облака выстроены «наоборот» — гипотеза для вопроса 1 (парадокс Симпсона).